TODO: This needs to be reworked.  The model we just created is identical to the first (champion) model, so the gate never triggers.

# Session 2 — The Deployment Gate

**Goal:** Understand the automated quality control that prevents bad models from reaching production.

The gate logic:
1. Load the **challenger** (newly trained model) from MLflow
2. Load the **champion** (current production model) from UC registry
3. Evaluate both on the holdout test set
4. If challenger F1 < champion F1 - threshold: **block deployment**
5. If challenger is within threshold: **promote to champion**

In [0]:
%run ../utils/config

In [0]:
%pip install /Volumes/{catalog}/00_shared/wheels/churn_model-0.1.0-py3-none-any.whl --quiet

In [0]:
dbutils.library.restartPython()

In [0]:
%run ../utils/config

In [0]:
dbutils.widgets.text("threshold", "0.05", "Max Allowed F1 Drop")

threshold = float(dbutils.widgets.get("threshold"))

In [0]:
import yaml, mlflow

config_path = "../common/config.yml"
with open(config_path) as f:
    config = yaml.safe_load(f)

from churn_model.evaluate import get_best_run

current_user  = spark.sql("SELECT current_user()").first()[0]
experiment_name = f"/Users/{current_user}/churn_{schema}"

challenger_run_id, model_type, challenger_metrics = get_best_run(
    experiment_name=experiment_name,
    metric="test_f1",
)
print(f"Challenger: {model_type}  F1={challenger_metrics['test_f1']:.4f}")

## Run the Gate

In [0]:
from churn_model.evaluate import evaluate_gate

model_name   = f"{catalog}.{schema}.churn_classifier"
champion_uri = f"models:/{model_name}@champion"
test_df      = spark.table(f"{catalog}.`00_shared`.telco_churn_holdout")

result = evaluate_gate(
    challenger_run_id=challenger_run_id,
    champion_model_uri=champion_uri,
    test_data=test_df,
    config=config,
    threshold=threshold,
)

print(f"\nDeploy? : {result.should_deploy}")
print(f"Reason  : {result.reason}")
print(f"\nChallenger: {result.challenger_metrics}")
print(f"Champion  : {result.champion_metrics}")
print(f"Delta     : {result.metrics_delta}")

## Simulate a Gate Failure

What happens when a bad model tries to get through?

Change the `threshold` widget to `0.001` (nearly impossible to beat)
and re-run the cell above. You'll see the gate reject deployment.

In [0]:
# Deliberately fail the gate
result_fail = evaluate_gate(
    challenger_run_id=challenger_run_id,
    champion_model_uri=champion_uri,
    test_data=test_df,
    config=config,
    threshold=0.001,  # Impossibly tight — will reject any challenger
)
print(f"Deploy? : {result_fail.should_deploy}")
print(f"Reason  : {result_fail.reason}")

## Discussion

- What threshold makes sense for this use case? (5%? 2%? 10%?)
- Should the gate block deployment or just alert?
- What happens to the challenger model run when the gate fails? (It stays in MLflow, nothing is deleted)
- How would you handle the case where no champion exists yet? (First deployment)

➡️ Next: `04_ab_canary_setup.ipynb`